# 배치 비교 시각화: 데이터 기반 통합 비교 플롯

기존 이미지를 이어붙이지 않고, **실제 데이터로부터 직접 정보를 추출하여 하나의 캔버스/축에서 3개 배치를 직접 비교**하는 분석용 노트북

- **Q1 (수명 분포)**: 하나의 박스플롯 축에 Batch 1, 2, 3을 나란히 배치 + 오버레이 히스토그램
- **Q2 (열화 곡선)**: 3개 배치의 대표 열화 궤적을 하나의 축에 오버레이 대조 + 무릎점(Knee point) 분포 비교
- **Q3 (ΔQ(V) 곡선)**: 단수명 그룹과 장수명 그룹의 전압별 ΔQ 곡선을 각각 하나의 축에 겹쳐서 비교
- **Q4 (C-rate vs 수명)**: 하나의 산점도 축에 3개 배치를 마커별로 함께 찍고 배치별 추세선 직접 비교
- **Q5 (상관관계)**: 피처별로 3개 배치의 상관계수를 나란히 묶은 **그룹드 바차트(Grouped Bar Chart)**로 일관된 핵심 신호 증명

In [14]:
import warnings
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Mac: AppleGothic 또는 Apple SD Gothic Neo)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14

CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR

DATA_DIR = ROOT / 'data'
RESULT_DIR = ROOT / 'results'

BATCH_FILES = {
    'Batch 1': DATA_DIR / 'batch_1_170512.mat',
    'Batch 2': DATA_DIR / 'batch_2_180220.mat',
    'Batch 3': DATA_DIR / 'batch_3_180412.mat',
}

all_cells = pd.read_csv(RESULT_DIR / 'all_batch_cell_features.csv')
print(f'한글 폰트 설정 완료 (AppleGothic) | 셀 요약 로드 완료: {len(all_cells)}개 셀')


한글 폰트 설정 완료 (AppleGothic) | 셀 요약 로드 완료: 139개 셀


## 1. [Q1] Cycle Life 분포: 단일 축 박스플롯 & 오버레이 히스토그램
- 하나의 축에 Batch 1, Batch 2, Batch 3의 박스플롯을 나란히 배치하여 중앙값과 수명 범위 차이를 극명하게 대조
- 우측에는 3개 배치의 수명 분포를 동일한 x축에 겹쳐서(Overlaid) 그려 수명 피크의 이동(Target Shift) 확인

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1) 좌측: 단일 축 배치별 박스플롯
batch_order = ['Batch 1', 'Batch 2', 'Batch 3']
colors = ['steelblue', 'tomato', 'seagreen']

box_data = [all_cells.loc[(all_cells['batch'] == b) & all_cells['cycle_life'].notna(), 'cycle_life'] for b in batch_order]

bp = axes[0].boxplot(box_data, patch_artist=True, tick_labels=batch_order,
                     medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[0].axhline(500, color='red', linestyle='--', linewidth=1.5, label='단수명 기준 (500)')
axes[0].axhline(1000, color='green', linestyle='--', linewidth=1.5, label='장수명 기준 (1,000)')
axes[0].set_title('배치별 Cycle Life 박스플롯', fontweight='bold')
axes[0].set_ylabel('Cycle Life')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(loc='upper right')

# 2) 우측: 오버레이 히스토그램
for b_name, col in zip(batch_order, colors):
    sub = all_cells.loc[(all_cells['batch'] == b_name) & all_cells['cycle_life'].notna(), 'cycle_life']
    axes[1].hist(sub, bins=np.arange(100, 2401, 100), color=col, alpha=0.45,
                 edgecolor='white', label=f'{b_name}')

axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axvline(500, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(1000, color='green', linestyle='--', linewidth=1.5)
axes[1].set_title('배치별 Cycle Life 히스토그램', fontweight='bold')
axes[1].set_xlabel('Cycle Life')
axes[1].set_ylabel('Cell Count')
axes[1].set_xlim(0, 2400)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(loc='upper right')

fig.suptitle('[Q1] 배치별 Cycle Life 분포 비교', fontweight='bold', fontsize=14, y=1.03)
fig.tight_layout()
fig.savefig(RESULT_DIR / 'q1_combined_cycle_life.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('저장 완료: q1_combined_cycle_life.png')


저장 완료: q1_combined_cycle_life.png


## 2. [Q2] 방전 용량($Q_d$) 열화 곡선 및 무릎점(Knee Point) 비교
- 원본 HDF5 파일에서 각 배치의 대표 셀(최단수명, 중앙값, 최장수명 셀) 궤적을 직접 추출하여 **하나의 축에 오버레이**
- 우측에는 셀별 Knee point 발생 사이클의 박스플롯을 배치하여, 가속 열화 시점의 극단적 차이 비교

In [16]:
def extract_rep_curves(batch_name, file_path):
    sub = all_cells[(all_cells['batch'] == batch_name) & all_cells['cycle_life'].notna()]
    sub_sorted = sub.sort_values('cycle_life')
    rep_indices = [sub_sorted.index[0], sub_sorted.index[len(sub)//2], sub_sorted.index[-1]]
    rep_cells = all_cells.loc[rep_indices]
    curves = []
    with h5py.File(file_path, 'r') as f:
        batch = f['batch']
        for _, row in rep_cells.iterrows():
            c_idx = int(row['cell_id'].split('c')[1])
            summary = f[batch['summary'][c_idx, 0]]
            cycle = np.asarray(summary['cycle']).squeeze()
            qd = np.asarray(summary['QDischarge']).squeeze()
            curves.append({
                'cell_id': row['cell_id'],
                'cycle_life': row['cycle_life'],
                'cycle': cycle,
                'qd': qd
            })
    return curves

b1_curves = extract_rep_curves('Batch 1', BATCH_FILES['Batch 1'])
b2_curves = extract_rep_curves('Batch 2', BATCH_FILES['Batch 2'])
b3_curves = extract_rep_curves('Batch 3', BATCH_FILES['Batch 3'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for curve in b1_curves:
    axes[0].plot(curve['cycle'], curve['qd'], color='steelblue', alpha=0.7, linewidth=1.5,
                 label='Batch 1' if curve == b1_curves[0] else "")
for curve in b2_curves:
    axes[0].plot(curve['cycle'], curve['qd'], color='tomato', alpha=0.7, linewidth=1.5,
                 label='Batch 2' if curve == b2_curves[0] else "")
for curve in b3_curves:
    axes[0].plot(curve['cycle'], curve['qd'], color='seagreen', alpha=0.7, linewidth=1.5,
                 label='Batch 3' if curve == b3_curves[0] else "")

axes[0].axhline(0.88, color='darkred', linestyle='--', linewidth=1.5, label='EOL (80% SOH)')
axes[0].set_title('대표 셀 방전 용량(QD) 열화 궤적', fontweight='bold')
axes[0].set_xlabel('Cycle')
axes[0].set_ylabel('QD (Ah)')
axes[0].set_ylim(0.7, 1.15)
axes[0].set_xlim(0, 2400)
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='lower left')

knee_data = [all_cells.loc[(all_cells['batch'] == b) & all_cells['capacity_95_cycle'].notna(), 'capacity_95_cycle'] for b in batch_order]
bp_k = axes[1].boxplot(knee_data, patch_artist=True, tick_labels=batch_order,
                       medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp_k['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[1].set_title('배치별 용량 95% 도달 시점 (capacity_95_cycle)', fontweight='bold')
axes[1].set_ylabel('Cycle (95% QD)')
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('[Q2] 배치별 QD 열화 곡선 및 Knee Point 비교', fontweight='bold', fontsize=14, y=1.03)
fig.tight_layout()
fig.savefig(RESULT_DIR / 'q2_combined_degradation.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('저장 완료: q2_combined_degradation.png')


저장 완료: q2_combined_degradation.png


## 3. [Q3] $\Delta Q(V)$ 곡선: 단수명 vs 장수명 단일 축 오버레이
- 각 배치의 원본 곡선에서 **단수명 그룹(하위 25%) 평균 곡선**과 **장수명 그룹(상위 25%) 평균 곡선** 직접 추출
- 좌측에는 3개 배치의 단수명 곡선을, 우측에는 3개 배치의 장수명 곡선을 하나의 축에 겹쳐서 비교

In [17]:
def extract_delta_q_means(batch_name, file_path):
    sub = all_cells[all_cells['batch'] == batch_name].dropna(subset=['cycle_life'])
    low_q, high_q = sub['cycle_life'].quantile([0.3, 0.7])
    short_ids = sub[sub['cycle_life'] <= low_q]['cell_id']
    long_ids = sub[sub['cycle_life'] >= high_q]['cell_id']
    short_curves, long_curves = [], []
    with h5py.File(file_path, 'r') as f:
        batch = f['batch']
        for i in range(batch['cycle_life'].shape[0]):
            c_id = f'b{batch_name[-1]}c{i}'
            if c_id in short_ids.values or c_id in long_ids.values:
                cycles = f[batch['cycles'][i, 0]]
                qd_10 = np.asarray(f[cycles['Qdlin'][9, 0]]).squeeze()
                qd_100 = np.asarray(f[cycles['Qdlin'][99, 0]]).squeeze()
                dq = qd_100 - qd_10
                if c_id in short_ids.values:
                    short_curves.append(dq)
                if c_id in long_ids.values:
                    long_curves.append(dq)
    short_mean = np.mean(short_curves, axis=0) if short_curves else np.zeros(1000)
    long_mean = np.mean(long_curves, axis=0) if long_curves else np.zeros(1000)
    return short_mean, long_mean

dq_means = {b: extract_delta_q_means(b, BATCH_FILES[b]) for b in batch_order}

voltage = np.linspace(2.0, 3.6, 1000)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for (b, (s_mean, _)), col in zip(dq_means.items(), colors):
    axes[0].plot(voltage, s_mean, color=col, linewidth=2, label=f'{b}')

axes[0].axhline(0, color='black', linestyle=':', alpha=0.6)
axes[0].axvspan(3.0, 3.4, color='yellow', alpha=0.15, label='3.0~3.4V 구간')
axes[0].set_title('단수명 셀 ΔQ(V) 곡선', fontweight='bold')
axes[0].set_xlabel('Voltage (V)')
axes[0].set_ylabel('Qd(100) - Qd(10) (Ah)')
axes[0].set_ylim(-0.12, 0.02)
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='lower left')

for (b, (_, l_mean)), col in zip(dq_means.items(), colors):
    axes[1].plot(voltage, l_mean, color=col, linewidth=2, linestyle='--', label=f'{b}')

axes[1].axhline(0, color='black', linestyle=':', alpha=0.6)
axes[1].set_title('장수명 셀 ΔQ(V) 곡선', fontweight='bold')
axes[1].set_xlabel('Voltage (V)')
axes[1].set_ylim(-0.12, 0.02)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='lower left')

fig.suptitle('[Q3] 배치별 ΔQ(V) 곡선 형상 비교', fontweight='bold', fontsize=14, y=1.03)
fig.tight_layout()
fig.savefig(RESULT_DIR / 'q3_combined_delta_q.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('저장 완료: q3_combined_delta_q.png')


저장 완료: q3_combined_delta_q.png


## 4. [Q4] 충전 속도(C-rate) vs 수명: 단일 축 산점도 & 배치별 추세선
- 하나의 축에 3개 배치의 셀들을 서로 다른 마커로 함께 찍고, 각 배치의 회귀 추세선을 그려 C-rate의 효과가 배치마다 엇갈림을 한눈에 증명

In [18]:
fig, ax = plt.subplots(figsize=(10, 6))
markers = ['o', 's', '^']

for b_name, col, m in zip(batch_order, colors, markers):
    sub = all_cells[(all_cells['batch'] == b_name) & all_cells['cycle_life'].notna() & all_cells['first_C_rate'].notna()]
    
    ax.scatter(sub['first_C_rate'], sub['cycle_life'], color=col, marker=m,
               s=70, alpha=0.75, edgecolors='black', linewidth=0.5, label=f'{b_name}')
    
    if len(sub) > 1:
        slope, intercept = np.polyfit(sub['first_C_rate'], sub['cycle_life'], 1)
        x_vals = np.linspace(sub['first_C_rate'].min(), sub['first_C_rate'].max(), 50)
        corr_s = sub['first_C_rate'].corr(sub['cycle_life'], method='spearman')
        ax.plot(x_vals, slope * x_vals + intercept, color=col, linestyle='--', linewidth=2,
                label=f'{b_name} (rho={corr_s:.2f})')

ax.set_title('[Q4] 충전 속도(First C-rate)와 수명 간의 관계 비교', fontweight='bold', fontsize=14, pad=12)
ax.set_xlabel('First C-rate', fontsize=11)
ax.set_ylabel('Cycle Life', fontsize=11)
ax.set_ylim(0, 2400)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', frameon=True, fontsize=10)

fig.tight_layout()
fig.savefig(RESULT_DIR / 'q4_combined_charge_policy.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('저장 완료: q4_combined_charge_policy.png')


저장 완료: q4_combined_charge_policy.png


## 5. [Q5] 초기 피처 상관계수 비교: 그룹드 바차트 (Grouped Bar Chart)
- 8개 초기 피처에 대해 **[Batch 1 / Batch 2 / Batch 3]의 상관계수 막대 3개를 나란히 묶어 단 하나의 차트**로 표현
- 다른 피처들은 배치마다 상관계수가 들쭉날쭉하지만, **`delta_q_var`는 3개 배치 모두에서 -0.7 이하로 일관되게 강력함**을 시각적으로 증명

In [19]:
corr1 = pd.read_csv(RESULT_DIR / 'batch_1_correlation.csv', index_col=0)['cycle_life'].drop('cycle_life')
corr2 = pd.read_csv(RESULT_DIR / 'batch_2_correlation.csv', index_col=0)['cycle_life'].drop('cycle_life')
corr3 = pd.read_csv(RESULT_DIR / 'batch_3_correlation.csv', index_col=0)['cycle_life'].drop('cycle_life')

features = corr1.index.tolist()
mean_corr = (corr1.abs() + corr2.abs() + corr3.abs()) / 3
features_sorted = mean_corr.sort_values(ascending=True).index.tolist()

y_pos = np.arange(len(features_sorted))
bar_height = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

rects1 = ax.barh(y_pos + bar_height, corr1.loc[features_sorted], bar_height,
                 color='steelblue', alpha=0.85, edgecolor='black', linewidth=0.5, label='Batch 1')
rects2 = ax.barh(y_pos, corr2.loc[features_sorted], bar_height,
                 color='tomato', alpha=0.85, edgecolor='black', linewidth=0.5, label='Batch 2')
rects3 = ax.barh(y_pos - bar_height, corr3.loc[features_sorted], bar_height,
                 color='seagreen', alpha=0.85, edgecolor='black', linewidth=0.5, label='Batch 3')

ax.axvline(0, color='black', linewidth=1)
ax.axvline(-0.7, color='crimson', linestyle='--', linewidth=1.5, label='|rho| >= 0.7')
ax.axvline(0.7, color='crimson', linestyle='--', linewidth=1.5)

highlight_idx = features_sorted.index('delta_q_var')
ax.axhspan(highlight_idx - 0.45, highlight_idx + 0.45, color='gold', alpha=0.2, zorder=0)

ax.set_yticks(y_pos)
ax.set_yticklabels(features_sorted, fontweight='bold', fontsize=11)
ax.set_xlim(-1.05, 1.05)
ax.set_xlabel('Spearman Correlation (rho)', fontsize=11)
ax.set_title('[Q5] 초기 피처와 Cycle Life 상관관계 비교', fontweight='bold', fontsize=14, pad=12)
ax.grid(axis='x', alpha=0.3)
ax.legend(loc='lower left', frameon=True, fontsize=10)

fig.tight_layout()
fig.savefig(RESULT_DIR / 'q5_combined_correlation.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('저장 완료: q5_combined_correlation.png')


저장 완료: q5_combined_correlation.png
